In [ ]:
import os
import time
import csv
import copy
import json
import re
import tifffile
from datetime import datetime, timezone
import matplotlib.pyplot as plt
import numpy as np
import numpy.linalg as la
from scipy import interpolate, signal
from scipy.optimize import least_squares
import pandas as pd
import seaborn as sns
import subprocess
import numpy as np
import pandas as pd
from pathlib import Path


In [3]:
BASE_DIR = Path(r"E:\permian_basin")     # 元の .txt と sample.json のある場所
TEMPLATE = BASE_DIR / "sample.json"      # ベース用テンプレ（各 .txt からNAME/GEOMETRY等を埋める）
BASE_JSON_DIR = BASE_DIR / "modtran_inputs"  # 中間生成（各 .txt → <stem>.json）
BASE_JSON_DIR.mkdir(exist_ok=True)

In [ ]:
for txt in sorted(BASE_DIR.glob("*.txt")): # ループでリセット(metaの辞書)
    meta = {}
    with txt.open("r", encoding="utf-8", errors="ignore") as f: 
        for line in f:
            if "=" not in line: # =がない行は無視
                continue
            k, v = line.split("=", 1) # key, valueに分割
            meta[k.strip()] = v.strip() # meta辞書に格納

    dt = datetime.strptime(meta["SceneCenterTime"], "%Y-%m-%dT%H:%M:%S.%fZ")
    IDAY   = dt.timetuple().tm_yday
    GMTIME = dt.hour + dt.minute / 60 + dt.second / 3600

    illum_elev = float(meta["IlluminationElevationAngleDegree"])
    illum_az   = float(meta["IlluminationAzimuthAngleDegree"])
    sc_elev    = float(meta["SpacecraftElevationAngleDegree"])
    sc_az      = float(meta["SpacecraftAzimuthAngleDegree"])
    emission   = float(meta["EmissionAngleDegree"]) if "EmissionAngleDegree" in meta else None

    BCKZEN = emission if emission is not None else 90.0 - sc_elev
    CKRANG = (sc_az + illum_az) % 360.0    
    PARM1  = illum_az                    
    PARM2  = illum_elev

    with TEMPLATE.open("r", encoding="utf-8") as f: # 各txtごとに同じsample.jsonより出発（よく分かってない）
        json_load = json.load(f)
    json_load['MODTRAN'][0]['MODTRANINPUT']['NAME'] = txt.stem  # 例: HSHL1G_N318W1030_20221030160056_20231127193054
    json_load['MODTRAN'][0]['MODTRANINPUT']['GEOMETRY']['IDAY']   = int(IDAY)
    json_load['MODTRAN'][0]['MODTRANINPUT']['GEOMETRY']['GMTIME'] = float(GMTIME)
    json_load['MODTRAN'][0]['MODTRANINPUT']['GEOMETRY']['BCKZEN'] = float(BCKZEN)
    json_load['MODTRAN'][0]['MODTRANINPUT']['GEOMETRY']['CKRANG'] = float(CKRANG)
    json_load['MODTRAN'][0]['MODTRANINPUT']['GEOMETRY']['PARM1']  = float(PARM1)
    json_load['MODTRAN'][0]['MODTRANINPUT']['GEOMETRY']['PARM2']  = float(PARM2)
    json_load['MODTRAN'][0]['MODTRANINPUT']['FILEOPTIONS']['CSVPRNT'] = f"{txt.stem}.csv"

    out_path = BASE_JSON_DIR / f"{txt.stem}.json" # 書き出し
    with out_path.open("w", encoding="utf-8") as f:
        json.dump(json_load, f, ensure_ascii=False, indent=2)
    print("Wrote:", out_path)

#------------------------------------------------------------------------------------   
# bg,test file 作成, これじゃなくてメタン濃度を変えたのを作りたいならここに入れる
def f2(x):  # 小数2桁に正規化
    return float(f"{x:.2f}")
# SURREF の範囲（0.00〜0.80, 0.01刻み）
start, stop, step = 0.00, 0.80, 0.01
n_surref = int(round((stop - start) / step)) + 1  # 81
# test用：先頭11層（0.0〜1.0 km）置換値（13通り）
head_values = [1.8 + d for d in [0.1, 0.3, 0.5, 1.0, 2.0, 3.0, 5.0, 7.5, 10.0, 15.0, 20.0, 25.0, 30.0]]
BACKGROUND_VALUE = 1.8
HEAD_LEN = 11
total_bg = 0
total_test = 0
for base_json in sorted(BASE_JSON_DIR.glob("*.json")):
    base_name = base_json.stem
    SCENE_ROOT   = BASE_DIR / base_name # 各txtごとに専用フォルダを作成（<stem>/bg と <stem>/test）
    SCENE_BG_DIR = SCENE_ROOT / "bg"
    SCENE_TEST_DIR = SCENE_ROOT / "test"
    SCENE_BG_DIR.mkdir(parents=True, exist_ok=True)
    SCENE_TEST_DIR.mkdir(parents=True, exist_ok=True)

    with base_json.open("r", encoding="utf-8") as f:
        base_data = json.load(f)

    for i in range(n_surref): # bg作成
        surref = f2(start + i * step)

        data = copy.deepcopy(base_data)
        mi = data["MODTRAN"][0]["MODTRANINPUT"]
        mi["SURFACE"]["SURREF"] = surref
        mi["FILEOPTIONS"]["CSVPRNT"] = f"{base_name}_bg_surref_{surref:0.2f}.csv"
        out_json = SCENE_BG_DIR / f"{surref:0.2f}.json"
        with out_json.open("w", encoding="utf-8") as fout:
            json.dump(data, fout, ensure_ascii=False, indent=2)

        total_bg += 1

    for i in range(n_surref): # test作成
        surref = f2(start + i * step)

        for v in head_values:
            v = f2(v)

            data = copy.deepcopy(base_data)
            mi = data["MODTRAN"][0]["MODTRANINPUT"]
            mi["SURFACE"]["SURREF"] = surref
            profiles = mi["ATMOSPHERE"]["PROFILES"]
            edited = False
            for prof in profiles:
                if prof.get("TYPE") == "PROF_CH4" and prof.get("UNITS") == "UNT_DPPMV":
                    arr = prof.get("PROFILE", [])
                    if len(arr) < HEAD_LEN:
                        raise ValueError(f"[{base_name}] PROFILE 長 {len(arr)} < {HEAD_LEN}")
                    prof["PROFILE"] = [v] * HEAD_LEN + [BACKGROUND_VALUE] * (len(arr) - HEAD_LEN)
                    edited = True
            if not edited:
                raise KeyError(f'[{base_name}] PROF_CH4 & UNT_DPPMV の PROFILE が見つかりません。')

            mi["FILEOPTIONS"]["CSVPRNT"] = f"{base_name}_test_surref_{surref:0.2f}_ch4_{v:0.2f}.csv"

            out_json = SCENE_TEST_DIR / f"{surref:0.2f}_ch4_{v:0.2f}.json"
            with out_json.open("w", encoding="utf-8") as fout:
                json.dump(data, fout, ensure_ascii=False, indent=2)

            total_test += 1

